In [1]:
import numpy as np
import pandas as pd

In [2]:
days = pd.read_csv('../Experimento_autoencoder_contrastive/attack_data/all_classes_normalized.csv')
days.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Idle Std,Idle Max,Idle Min,ohe_port_hf,ohe_port_mf,ohe_port_lf,ohe_prot_6,ohe_prot_17,ohe_prot_0,Label
0,0.999814,0.000013,0.000024,0.000038,1.905853e-06,0.025688,0.000000,0.017072,0.051853,0.004573,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign
1,0.999814,0.000003,0.000016,0.000006,9.337401e-07,0.002045,0.028082,0.006796,0.000000,0.001120,...,0.0,0.0,0.0,1,0,0,0,1,0,Benign
2,0.999813,0.000003,0.000008,0.000003,0.000000e+00,0.001895,0.000000,0.003149,0.006049,0.000000,...,0.0,0.0,0.0,0,0,1,1,0,0,Benign
3,0.999813,0.000000,0.000008,0.000002,3.069830e-07,0.001596,0.021918,0.005304,0.000000,0.000737,...,0.0,0.0,0.0,1,0,0,0,1,0,Benign
4,0.999816,0.000023,0.000057,0.000077,9.139141e-06,0.028182,0.000000,0.021381,0.042983,0.017634,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign


In [3]:
days['Label'].value_counts()

Label
Benign                      803354
DDOS attack-HOIC            686012
DDoS attacks-LOIC-HTTP      576191
DoS attacks-Hulk            461912
Bot                         286191
FTP-BruteForce              193354
SSH-Bruteforce              187589
Infilteration               160639
DoS attacks-SlowHTTPTest    139890
DoS attacks-GoldenEye        41508
DoS attacks-Slowloris        10990
DDOS attack-LOIC-UDP          1730
Brute Force -Web               611
Brute Force -XSS               230
SQL Injection                   87
Name: count, dtype: int64

In [3]:
days['Label'] = days['Label'].replace('FTP-BruteForce','BruteForce')
days['Label'] = days['Label'].replace('SSH-Bruteforce','BruteForce')
days['Label'] = days['Label'].replace('DoS attacks-GoldenEye','DoS')
days['Label'] = days['Label'].replace('DoS attacks-Slowloris','DoS')
days['Label'] = days['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
days['Label'] = days['Label'].replace('DoS attacks-Hulk','DoS')
days['Label'] = days['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
days['Label'] = days['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
days['Label'] = days['Label'].replace('DDOS attack-HOIC','DDoS')
days['Label'] = days['Label'].replace('Brute Force -Web','Web')
days['Label'] = days['Label'].replace('Brute Force -XSS','Web')
days['Label'] = days['Label'].replace('SQL Injection','Web')

In [4]:
labels = days['Label'].unique()

labels

array(['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS',
       'DDOS attack-HOIC', 'DDoS attacks-LOIC-HTTP',
       'DDOS attack-LOIC-UDP', 'DoS attacks-GoldenEye',
       'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest',
       'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration',
       'SQL Injection', 'SSH-Bruteforce'], dtype=object)

In [5]:
days[-6:]

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Idle Std,Idle Max,Idle Min,ohe_port_hf,ohe_port_mf,ohe_port_lf,ohe_prot_6,ohe_prot_17,ohe_prot_0,Label
3550282,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
3550283,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
3550284,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
3550285,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
3550286,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
3550287,0.999813,0.0,0.000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce


In [6]:
test = pd.DataFrame()
train_val = pd.DataFrame()
for l in labels:
    shuf = days[days['Label'] == l].sample(frac=1)
    train_val = pd.concat([train_val,shuf[:-(shuf.shape[0]//5)]], ignore_index=True)
    test = pd.concat([test,shuf[-(shuf.shape[0]//5):]], ignore_index=True)

test

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Idle Std,Idle Max,Idle Min,ohe_port_hf,ohe_port_mf,ohe_port_lf,ohe_prot_6,ohe_prot_17,ohe_prot_0,Label
0,0.999814,0.000006,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign
1,0.999816,0.000023,0.000057,0.000084,1.011125e-05,0.032971,0.000000,0.023370,0.050121,0.018002,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign
2,0.999821,0.000026,0.000065,0.000085,1.011125e-05,0.033769,0.000000,0.021068,0.049195,0.018002,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign
3,0.999813,0.000003,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,1,0,0,1,0,0,Benign
4,0.999813,0.000003,0.000016,0.000007,7.930395e-07,0.002294,0.031507,0.007624,0.000000,0.000952,...,0.0,0.0,0.0,1,0,0,0,1,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710047,0.999814,0.000068,0.000162,0.000144,1.704395e-05,0.031923,0.000000,0.014646,0.031040,0.014979,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
710048,0.999814,0.000068,0.000179,0.000142,1.704395e-05,0.031923,0.000000,0.014405,0.030999,0.014979,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
710049,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce
710050,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,1,0,0,1,0,0,SSH-Bruteforce


In [7]:
train_val['Label'].value_counts()

Label
Benign                      642684
DDOS attack-HOIC            548810
DDoS attacks-LOIC-HTTP      460953
DoS attacks-Hulk            369530
Bot                         228953
FTP-BruteForce              154684
SSH-Bruteforce              150072
Infilteration               128512
DoS attacks-SlowHTTPTest    111912
DoS attacks-GoldenEye        33207
DoS attacks-Slowloris         8792
DDOS attack-LOIC-UDP          1384
Brute Force -Web               489
Brute Force -XSS               184
SQL Injection                   70
Name: count, dtype: int64

In [8]:
test['Label'].value_counts()

Label
Benign                      160670
DDOS attack-HOIC            137202
DDoS attacks-LOIC-HTTP      115238
DoS attacks-Hulk             92382
Bot                          57238
FTP-BruteForce               38670
SSH-Bruteforce               37517
Infilteration                32127
DoS attacks-SlowHTTPTest     27978
DoS attacks-GoldenEye         8301
DoS attacks-Slowloris         2198
DDOS attack-LOIC-UDP           346
Brute Force -Web               122
Brute Force -XSS                46
SQL Injection                   17
Name: count, dtype: int64

In [9]:
train = pd.DataFrame()
val = pd.DataFrame()
for l in labels:
    shuf = train_val[train_val['Label'] == l].sample(frac=1)
    train = pd.concat([train,shuf[:-(shuf.shape[0]//5)]], ignore_index=True)
    val = pd.concat([val,shuf[-(shuf.shape[0]//5):]], ignore_index=True)

train

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Idle Std,Idle Max,Idle Min,ohe_port_hf,ohe_port_mf,ohe_port_lf,ohe_prot_6,ohe_prot_17,ohe_prot_0,Label
0,0.999813,0.000003,0.000016,0.000005,1.560497e-06,0.001796,0.024658,0.005967,0.000000,0.001872,...,0.000000e+00,0.00000,0.000000,1,0,0,0,1,0,Benign
1,0.999913,0.000032,0.000073,0.000030,4.803006e-06,0.019802,0.000000,0.006072,0.026909,0.011525,...,7.590897e-08,0.00001,0.001568,1,0,0,1,0,0,Benign
2,0.999813,0.000013,0.000016,0.000069,2.020972e-06,0.046638,0.000000,0.030994,0.094140,0.004850,...,0.000000e+00,0.00000,0.000000,0,0,1,1,0,0,Benign
3,0.999813,0.000000,0.000008,0.000003,3.453559e-07,0.001895,0.026027,0.006298,0.000000,0.000829,...,0.000000e+00,0.00000,0.000000,1,0,0,0,1,0,Benign
4,0.999813,0.000006,0.000000,0.000002,0.000000e+00,0.001546,0.000000,0.001713,0.004029,0.000000,...,0.000000e+00,0.00000,0.000000,0,0,1,1,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2272191,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.00000,0.000000,1,0,0,1,0,0,SSH-Bruteforce
2272192,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.00000,0.000000,1,0,0,1,0,0,SSH-Bruteforce
2272193,0.999814,0.000068,0.000179,0.000142,1.704395e-05,0.031923,0.000000,0.014405,0.030999,0.014979,...,0.000000e+00,0.00000,0.000000,1,0,0,1,0,0,SSH-Bruteforce
2272194,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.00000,0.000000,1,0,0,1,0,0,SSH-Bruteforce


In [10]:
train['Label'].value_counts()

Label
Benign                      514148
DDOS attack-HOIC            439048
DDoS attacks-LOIC-HTTP      368763
DoS attacks-Hulk            295624
Bot                         183163
FTP-BruteForce              123748
SSH-Bruteforce              120058
Infilteration               102810
DoS attacks-SlowHTTPTest     89530
DoS attacks-GoldenEye        26566
DoS attacks-Slowloris         7034
DDOS attack-LOIC-UDP          1108
Brute Force -Web               392
Brute Force -XSS               148
SQL Injection                   56
Name: count, dtype: int64

In [11]:
val['Label'].value_counts()

Label
Benign                      128536
DDOS attack-HOIC            109762
DDoS attacks-LOIC-HTTP       92190
DoS attacks-Hulk             73906
Bot                          45790
FTP-BruteForce               30936
SSH-Bruteforce               30014
Infilteration                25702
DoS attacks-SlowHTTPTest     22382
DoS attacks-GoldenEye         6641
DoS attacks-Slowloris         1758
DDOS attack-LOIC-UDP           276
Brute Force -Web                97
Brute Force -XSS                36
SQL Injection                   14
Name: count, dtype: int64

In [12]:
train = train.sample(frac=1)
val = val.sample(frac=1)
test = test.sample(frac=1)

In [13]:
train.to_csv('train_CADE_cod.csv', index=False)
val.to_csv('val_CADE_cod.csv', index=False)
test.to_csv('test_CADE_cod.csv', index=False)